In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also run:
```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


In [8]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I’m not seeing anything about **Olama** in the FAQ context.

If you meant **running the course locally**, the FAQ says you can do that if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. It also says to **document your setup** and keep it **reproducible**.

If you meant something else by “Olama,” send me the exact tool/name and I’ll check the context again.


In [9]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes — you can usually join a course after discovering it, as long as it’s still open for enrollment.\n\nTo check, look for:\n- an **Enroll / Join / Register** button\n- the **course start date**\n- any **deadline or enrollment limit**\n- whether you need an **invite code** or approval\n\nIf you want, send me the **course name or link**, and I can help you figure out the exact steps.'

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [13]:
len(response.output)

1

In [14]:
call = response.output[0]

In [15]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll can I join after course started registration FAQ"}', call_id='call_h7urwpPBsAdjIq3WQplApvIF', name='search', type='function_call', id='fc_0ed90f93e1c2efb3006a364f3214e0819d83e226c89eace8b6', namespace=None, status='completed')

In [16]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late enroll can I join after course started registration FAQ'}

In [17]:
call.name

'search'

In [18]:
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [21]:
messages.append(call)

In [22]:
messages.append(function_call_output)

In [23]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll can I join after course started registration FAQ"}', call_id='call_h7urwpPBsAdjIq3WQplApvIF', name='search', type='function_call', id='fc_0ed90f93e1c2efb3006a364f3214e0819d83e226c89eace8b6', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_h7urwpPBsAdjIq3WQplApvIF',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mod

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [25]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just learning, you can start anytime.


In [26]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 42)

In [27]:
def calculate_openai_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float = 0.25,
    output_price_per_million: float = 2.00,
) -> float:
    """
    Calculate OpenAI API cost in USD.

    Args:
        input_tokens: Number of input tokens.
        output_tokens: Number of output tokens.
        input_price_per_million: USD per 1M input tokens.
        output_price_per_million: USD per 1M output tokens.

    Returns:
        Total cost in USD.
    """
    input_cost = (input_tokens / 1_000_000) * input_price_per_million
    output_cost = (output_tokens / 1_000_000) * output_price_per_million
    return input_cost + output_cost

In [28]:
cost = calculate_openai_cost(779, 37)
print(f"${cost:.8f}")

$0.00026875


In [29]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [31]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [32]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course enroll discover course can I join it"}
function_call: search {"query":"course enrollment discovered course can I join"}
function_call: search {"query":"how to join course after it starts enrollment access"}


In [33]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll discover course can I join it"}', call_id='call_hrWTy487srIjzT7pFeeDMSvD', name='search', type='function_call', id='fc_04d0ad3953057778006a364f348d3881a0a173c8579ee5a558', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment discovered course can I join"}', call_id='call_Kl23v8CWqbAmihttmTBAj7Ny',

In [34]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered late enroll registration can I join course FAQ"}
function_call: search {"query":"late enrollment discovered course can I join student FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. 

If you want a certificate, you’ll need to submit your project while submissions are still being accepted. If you’re just learning, you can start anytime from the course materials.

If you want, I can also help you with the next steps for getting started.


In [35]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [37]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment discovered the course can I join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still being accepted. If you’re just following along for learning, you can start anytime.

If you want, I can also explain how the certificate/peer-review process works or how to get started with the course materials.


In [38]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still being accepted. If you’re just following along for learning, you can start anytime.\n\nIf you want, I can also explain how the certificate/peer-review process works or how to get started with the course materials.'

In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening what is queen's gambit"}
iteration #2...
function_call: search {"query":"gambit queen's gambit chess opening FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this isn’t a course-related question.

If you meant something else related to the course, feel free to ask, and I can help with that. Are there other areas you want to explore?
